# 01 — Method A: **BF16 LoRA** 베이스라인 (KorQuAD)

발표(Serve Track: LoRA → INT4 PTQ/QAT → vLLM)용 **3-way 양자화 비교**의 기준선.

| 방법 | 무엇 | 언제 |
|---|---|---|
| **A = BF16 LoRA** (이 노트북) | 풀정밀(BF16) 베이스에 LoRA 어댑터 학습 후 **머지** | 품질 상한 기준선 |
| B = INT4 PTQ (02) | A의 머지 모델을 **사후** 4bit 양자화 | 학습 없이 압축 |
| C = INT4 QAT (03) | 4bit **인식 학습** | 양자화 오차를 학습으로 보정 |

A는 B·C의 **입력(머지 BF16 모델)**이자 품질 기준이므로 먼저 확정합니다.

> ## ⚙️ 실행 모드 배너 — 이 노트북은 **CPU 스모크**로 실행됨
>
> **왜:** 대상 Azure 구독(`ME-MngEnvMCAP756842-hjeon-1`)에 **모던 GPU 쿼터가 0**이고,
> H100·A100·A10 **쿼터 증설 요청을 제출**했으나 승인되지 않았습니다.
> (요청 IDs — H100 `e9245f08`: Failed, A10 `d9addbec`: Failed, A100 `bd193591`: InProgress.
> 과거 동일 패밀리 요청도 모두 Failed — 스폰서 구독의 GPU 잠금으로 판단.)
>
> **결과:** "쿼터 중 있는 걸로 처리"하라는 지시에 따라, 파이프라인을 **동일 코드 경로**로
> 소형 모델(`Qwen/Qwen2.5-0.5B-Instruct`) + 200개 서브셋 + 12스텝의 **CPU 스모크**로
> 실제 실행해 모든 블록이 통과함을 **실 출력**으로 증명합니다.
>
> **프로덕션(스펙) 실행:** GPU VM에서 `config.yaml`의 `compute.mode: gpu`(base=`Qwen/Qwen3-1.7B`,
> `train.backend: unsloth`, BF16, full data)로 **동일 셀**을 실행하면 스펙의 A100/H100 BF16
> 베이스라인이 나옵니다. 코드는 그대로, 설정만 다릅니다.

### 0) 부트스트랩 & 버전 고정 (재현성)
`quantization/` 공용 모듈을 import하고, 정확한 버전을 `results/env_A.json`에 기록.

In [1]:
import os, sys
here = os.getcwd()
for cand in [here, os.path.dirname(here), os.path.join(here, "pdf_qa_extraction"),
             os.path.dirname(os.path.dirname(here))]:
    if os.path.isdir(os.path.join(cand, "quantization")):
        if cand not in sys.path:
            sys.path.insert(0, cand)
        os.chdir(cand)
        break
print("cwd:", os.getcwd())

cwd: /ai-work/copilot/PDF2LLM-Tuning-Studio/pdf_qa_extraction


In [2]:
import json, platform, torch, transformers, trl, peft, datasets
env = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "trl": trl.__version__,
    "peft": peft.__version__,
    "datasets": datasets.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
}
os.makedirs("quantization/results", exist_ok=True)
with open("quantization/results/env_A.json", "w", encoding="utf-8") as fh:
    json.dump(env, fh, ensure_ascii=False, indent=2)
env

/home/dev/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.10.12',
 'torch': '2.13.0+cu130',
 'transformers': '5.14.1',
 'trl': '1.9.0',
 'peft': '0.19.1',
 'datasets': '5.0.0',
 'cuda_available': False,
 'device': 'cpu'}

### 1) 설정 로드
`compute.mode: cpu` 스모크 오버라이드가 적용됩니다(소형 모델·서브셋·12스텝). GPU VM에선 `mode: gpu`로 두면 됩니다.

In [3]:
from quantization.data_korquad import load_config, load_korquad, to_hf_text_dataset
cfg = load_config(force_mode='cpu')   # GPU VM: load_config() (mode=gpu from yaml)
print('base :', cfg['base_model']['selected'])
print('mode :', cfg['compute']['mode'], '| backend:', cfg['train']['backend'],
      '| precision:', cfg['train']['precision'])
print('lora :', cfg['lora'])
print('data :', {k: cfg['data'][k] for k in ['dataset','seed','eval_size','train_subset','max_seq_len']})

base : Qwen/Qwen2.5-0.5B-Instruct
mode : cpu | backend: hf | precision: fp32
lora : {'r': 16, 'alpha': 32, 'dropout': 0.0, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']}
data : {'dataset': 'KorQuAD/squad_kor_v1', 'seed': 42, 'eval_size': 20, 'train_subset': 200, 'max_seq_len': 384}


### 2) 데이터 — KorQuAD → 생성 instruction 포맷
런타임 다운로드(레포 미커밋). 고정 seed로 held-out val 슬라이스(A/B/C 동일).

In [4]:
data = load_korquad(cfg)
print('train:', len(data['train']), '| eval(held-out):', len(data['eval']))
ex = data['eval'][0]
print('\n--- prompt ---\n' + ex.prompt)
print('--- gold answers ---', ex.answers)

train: 200 | eval(held-out): 20

--- prompt ---
아래 문맥을 읽고 질문에 답하세요.

[문맥]
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교통비 절감 효과로 이용객이 늘어나자 버스, 지하철 회사의 수입이 증가하였다. 초기엔 불편을 토로하던 시민들도 정착 후에는 바뀐 교통체계를 지지하는 사람들이 많아졌고 이는 이명박의 대중 인기 증가에 큰 보탬이 되었다. 이에 힘입어 이명박은 서울시장 퇴임 후 대선 후보에 올라 당선되기에 이른다.

[질문] 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
[답]

--- gold answers --- ['대중교통체계']


### 3) EM/F1 지표 자가검증 (KorQuAD 공식 · 한국어 char-level)
SQuAD 나이브 EM이 아니라 KorQuAD 정규화 + 문자단위 F1임을 확인.

In [5]:
import quantization.eval_qa as E
E._selftest()
print('공식 정규화 예:', repr(E.normalize_answer('세종대왕! (조선)')))

[eval selftest] EM/F1 over 3 pairs: {'exact_match': 66.66666666666667, 'f1': 90.90909090909092, 'n': 3}
[eval selftest] OK
공식 정규화 예: '세종대왕 조선'


### 4) 학습 — BF16 LoRA (r=16, α=32, attn+MLP)
스모크: 0.5B·12스텝. 산출물 = LoRA 어댑터 + **머지 모델**(`artifacts/A_bf16/`, Part 2 입력).

In [6]:
from quantization.train_lora import train
train_log = train(cfg)
train_log

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Adding EOS to train dataset: 100%|██████████| 200/200 [00:00<00:00, 22499.22 examples/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing train dataset:  73%|███████▎  | 146/200 [00:00<00:00, 1450.59 examples/s]

...

Dropping fully masked examples from train dataset: 100%|██████████| 200/200 [00:00<00:00, 3610.13 examples/s]


[RANK 0] Detected kernel version 3.10.102, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.716959


{'base_model': 'Qwen/Qwen2.5-0.5B-Instruct',
 'backend': 'hf',
 'precision': 'fp32',
 'mode': 'cpu',
 'n_train': 200,
 'train_seconds': 115.69,
 'train_loss': 2.6426029205322266,
 'global_step': 12,
 'adapter_dir': 'quantization/artifacts/A_bf16_adapter',
 'merged_dir': 'quantization/artifacts/A_bf16'}

### 5) 동작 데모 (필수) — held-out 질문 **1개**
튜닝된 머지 모델을 로드해 실제로 답을 생성합니다(“동작한다”를 눈으로).

In [7]:
model, tok = E.load_model_for_eval(cfg['paths']['method_a_dir'], cfg['train']['precision'])
demo = data['eval'][0]
gen = E.generate_answers(model, tok, [demo.prompt], max_new_tokens=32, batch_size=1)
print('[질문]', demo.question)
print('[정답]', demo.answers)
print('[모델 답]', gen['answers'][0])

[질문] 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
[정답] ['대중교통체계']
[모델 답] 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입


### 6) 수치 — EM/F1 · perplexity · 크기 · VRAM · tok/s
`results/A_bf16_metrics.json` 저장 + 3-way 표 첫 행(`three_way_table.json`) append.

In [8]:
res = E.evaluate_model(model, tok, data['eval'], method='A_bf16',
                       base_model=cfg['base_model']['selected'],
                       model_dir=cfg['paths']['method_a_dir'],
                       max_new_tokens=cfg['eval']['max_new_tokens'],
                       batch_size=cfg['eval']['batch_size'],
                       ppl_samples=cfg['eval']['ppl_samples'],
                       precision=cfg['train']['precision'],
                       notes=f"mode={cfg['compute']['mode']} backend={cfg['train']['backend']}")
E.write_metrics(res, cfg['paths']['results_dir'])
E.append_to_table(res, cfg['paths']['results_dir'])
from dataclasses import asdict
row = asdict(res)
print('A (BF16 LoRA) — 3-way 표 첫 행')
for k in ['method','base_model','exact_match','f1','perplexity','size_gb','peak_vram_gb','tok_per_s','precision']:
    print(f'  {k:14}: {row[k]}')

A (BF16 LoRA) — 3-way 표 첫 행
  method        : A_bf16
  base_model    : Qwen/Qwen2.5-0.5B-Instruct
  exact_match   : 30.0
  f1            : 45.177
  perplexity    : 15.685
  size_gb       : 1.8511
  peak_vram_gb  : None
  tok_per_s     : 5.63
  precision     : fp32


### 7) 다음 단계 (Part 2)
머지 BF16 모델(`artifacts/A_bf16/`)을 입력으로 **B. INT4 PTQ**(`02`)와 **C. INT4 QAT**(`03`)를 동일 템플릿·동일 eval로 실행해 3-way 표를 완성하고 vLLM 서빙 벤치로 잇습니다.

> 프로덕션 재현: GPU VM에서 `compute.mode: gpu`로 이 노트북을 재실행하면 표의 A 행이 Qwen3-1.7B BF16(A100/H100) 수치로 채워집니다.